# 05 — Ablation Studies

Three controlled ablations using the DualEncoder + BM25 hard-negative baseline:

| Ablation | Variable | Values tested |
|----------|----------|---------------|
| A | Pooling strategy | mean (default) vs CLS |
| B | InfoNCE temperature τ | 0.05 (default), 0.1, 0.2 |
| C | Random seed stability | seeds 42, 123, 456 |

**Rule**: Change **one** variable at a time; keep all other hyperparameters fixed.

In [ ]:
import sys, os
sys.path.insert(0, '..')

import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
from pathlib import Path

from src.encoder import DualEncoder, load_encoder
from src.dense_retriever import DenseRetriever
from src.metrics import compute_metrics, aggregate, print_metrics_table

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
RESULTS = Path('../results')
RESULTS.mkdir(parents=True, exist_ok=True)

# Load data once
corpus_df   = pd.read_parquet('../data/corpus.parquet')
test_df     = pd.read_parquet('../data/test.parquet')
corpus_ids  = corpus_df['product_id'].tolist()
corpus_docs = corpus_df['product_doc'].tolist()
query_texts = test_df['review_text'].tolist()
print(f'Test: {len(test_df):,} | Corpus: {len(corpus_df):,}')

In [ ]:
def evaluate_checkpoint(ckpt_path, label, model_type='dual', pooling='mean'):
    """Load a checkpoint, encode corpus/queries, and return aggregated metrics."""
    if not os.path.exists(ckpt_path):
        print(f'[SKIP] {label}: checkpoint not found at {ckpt_path}')
        return None

    print(f'Evaluating: {label}')
    model = load_encoder(ckpt_path).to(DEVICE)
    model.eval()

    corpus_embs = model.encode_docs(corpus_docs, batch_size=8)
    retriever   = DenseRetriever(corpus_ids, corpus_embs)
    query_embs  = model.encode_queries(query_texts, batch_size=32)
    results     = retriever.batch_retrieve(query_embs, k=10)

    metrics_list = []
    for i, (_, row) in enumerate(test_df.iterrows()):
        retrieved = [pid for pid, _ in results[i]]
        metrics_list.append(compute_metrics(retrieved, row['product_id']))

    agg = aggregate(metrics_list)
    print_metrics_table(agg, title=label)
    return agg

## Ablation A — Pooling Strategy: Mean vs CLS

**Mean pooling**: Attention-mask-weighted average of all token hidden states.  
**CLS pooling**: Use only the `[CLS]` token representation.

Prior work (Reimers & Gurevych 2019, SBERT) shows mean pooling outperforms CLS for sentence similarity.  
This ablation verifies that finding holds on our retrieval task.

In [ ]:
print('Training commands (run from project root):')
print()
print('# Ablation A: CLS pooling')
print('python train.py --model-type dual --neg-mode bm25 --pooling cls \\')
print('  --output-dir artifacts/models/dual_hardneg_cls/ --seed 42')
print()

ablation_a_results = {}

# Mean pooling (already trained as default)
mean_agg = evaluate_checkpoint(
    '../artifacts/models/dual_hardneg_seed42/best_model',
    'Dual + hard-neg, MEAN pooling'
)
if mean_agg:
    ablation_a_results['Mean'] = mean_agg

# CLS pooling
cls_agg = evaluate_checkpoint(
    '../artifacts/models/dual_hardneg_cls/best_model',
    'Dual + hard-neg, CLS pooling'
)
if cls_agg:
    ablation_a_results['CLS'] = cls_agg

In [ ]:
# Plot Ablation A
if len(ablation_a_results) >= 2:
    metrics_keys = ['ndcg@10', 'recall@10', 'mrr', 'recall@1']
    x = np.arange(len(metrics_keys))
    width = 0.35

    fig, ax = plt.subplots(figsize=(9, 5))
    bars_mean = ax.bar(x - width/2,
                       [ablation_a_results['Mean'].get(k, 0) for k in metrics_keys],
                       width, label='Mean pooling', color='steelblue')
    bars_cls  = ax.bar(x + width/2,
                       [ablation_a_results['CLS'].get(k, 0) for k in metrics_keys],
                       width, label='CLS pooling', color='orange', alpha=0.85)

    ax.set_xticks(x)
    ax.set_xticklabels(metrics_keys)
    ax.set_ylabel('Score')
    ax.set_ylim(0, 1.0)
    ax.set_title('Ablation A: Pooling Strategy (Dual Encoder + BM25 Hard-Neg)')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)

    # Annotate bars
    for bar in bars_mean:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)
    for bar in bars_cls:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)

    plt.tight_layout()
    plt.savefig('../results/ablation_pooling.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: results/ablation_pooling.png')

    ndcg_diff = ablation_a_results['Mean'].get('ndcg@10', 0) - ablation_a_results['CLS'].get('ndcg@10', 0)
    print(f'\nNDCG@10 delta (mean - CLS): {ndcg_diff:+.4f}')
    if ndcg_diff > 0:
        print('Mean pooling is BETTER — consistent with SBERT findings.')
    elif ndcg_diff < -0.005:
        print('CLS pooling is better on this task — interesting deviation from standard findings.')
    else:
        print('Pooling strategies are roughly equivalent (<0.005 NDCG@10 difference).')
else:
    print('Train both pooling variants first.')

## Ablation B — InfoNCE Temperature τ

Temperature τ controls the sharpness of the softmax distribution:
- **Low τ (0.05)**: Sharp distribution — model focuses intensely on the hardest negatives
- **High τ (0.2)**: Flat distribution — gentler learning signal, risk of underfitting

SimCSE (Gao et al. 2021) found τ=0.05 optimal; we verify on retrieval.

In [ ]:
print('Training commands:')
for tau in [0.1, 0.2]:
    print(f'python train.py --model-type dual --neg-mode bm25 --temperature {tau} \\')
    print(f'  --output-dir artifacts/models/dual_hardneg_tau{str(tau).replace(".","")}/ --seed 42')
    print()

ablation_b_results = {}
tau_configs = [
    (0.05, '../artifacts/models/dual_hardneg_seed42/best_model'),
    (0.10, '../artifacts/models/dual_hardneg_tau010/best_model'),
    (0.20, '../artifacts/models/dual_hardneg_tau020/best_model'),
]

for tau, ckpt in tau_configs:
    agg = evaluate_checkpoint(ckpt, f'Dual + hard-neg, τ={tau}')
    if agg:
        ablation_b_results[tau] = agg

In [ ]:
# Plot Ablation B — temperature sweep
if len(ablation_b_results) >= 2:
    taus   = sorted(ablation_b_results.keys())
    ndcgs  = [ablation_b_results[t].get('ndcg@10', 0)  for t in taus]
    r10s   = [ablation_b_results[t].get('recall@10', 0) for t in taus]
    mrrs   = [ablation_b_results[t].get('mrr', 0)       for t in taus]

    fig, axes = plt.subplots(1, 3, figsize=(13, 4))
    for ax, vals, ylabel in zip(axes, [ndcgs, r10s, mrrs], ['NDCG@10', 'Recall@10', 'MRR']):
        ax.plot([str(t) for t in taus], vals, marker='o', lw=2, color='steelblue')
        ax.set_xlabel('Temperature τ')
        ax.set_ylabel(ylabel)
        ax.set_title(f'{ylabel} vs Temperature')
        ax.grid(alpha=0.3)
        for x_val, y_val in zip(range(len(taus)), vals):
            ax.annotate(f'{y_val:.4f}', (x_val, y_val), textcoords='offset points',
                        xytext=(0, 8), ha='center', fontsize=9)

    plt.suptitle('Ablation B: InfoNCE Temperature τ (Dual Encoder + BM25 Hard-Neg)', y=1.02)
    plt.tight_layout()
    plt.savefig('../results/ablation_temperature.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: results/ablation_temperature.png')

    best_tau = taus[np.argmax(ndcgs)]
    print(f'\nBest τ by NDCG@10: {best_tau}')
    print('Note: Lower temperatures can cause training instability if too aggressive.')
else:
    print('Train all temperature variants first.')

## Ablation C — Random Seed Stability

Verify results are reproducible across different random seeds.  
High variance would indicate the model is unstable and results may not generalize.

In [ ]:
print('Training commands:')
for seed in [123, 456]:
    print(f'python train.py --model-type dual --neg-mode bm25 --seed {seed} \\')
    print(f'  --output-dir artifacts/models/dual_hardneg_seed{seed}/')
    print()

seed_results = {}
seed_configs = [
    (42, '../artifacts/models/dual_hardneg_seed42/best_model'),
    (123, '../artifacts/models/dual_hardneg_seed123/best_model'),
    (456, '../artifacts/models/dual_hardneg_seed456/best_model'),
]

for seed, ckpt in seed_configs:
    agg = evaluate_checkpoint(ckpt, f'Dual + hard-neg, seed={seed}')
    if agg:
        seed_results[seed] = agg

In [ ]:
# Compute mean ± std across seeds
if len(seed_results) >= 2:
    metrics_keys = ['ndcg@10', 'recall@10', 'mrr', 'recall@1']
    print('Multi-seed results:')
    print(f'{"Seed":<8} ' + ' '.join(f'{k:>12}' for k in metrics_keys))
    print('-' * 60)
    for seed in sorted(seed_results.keys()):
        row = ' '.join(f'{seed_results[seed].get(k, 0):>12.4f}' for k in metrics_keys)
        print(f'{seed:<8} {row}')

    print('-' * 60)
    for metric in metrics_keys:
        vals = [seed_results[s].get(metric, 0) for s in seed_results]
        mean_v = np.mean(vals)
        std_v  = np.std(vals)
        print(f'{metric:<12}: {mean_v:.4f} ± {std_v:.4f}')

    # Stability assessment
    ndcg_vals = [seed_results[s].get('ndcg@10', 0) for s in seed_results]
    cv = np.std(ndcg_vals) / np.mean(ndcg_vals) * 100  # coefficient of variation
    print(f'\nNDCG@10 CV (std/mean): {cv:.2f}%')
    if cv < 1.0:
        print('Results are HIGHLY STABLE (CV < 1%).')
    elif cv < 3.0:
        print('Results are STABLE (CV < 3%).')
    else:
        print('HIGH VARIANCE (CV ≥ 3%) — consider averaging over more seeds.')
else:
    print('Train at least 2 seeds first.')

In [ ]:
# Plot seed stability
if len(seed_results) >= 2:
    metrics_keys = ['ndcg@10', 'recall@10', 'mrr', 'recall@1']
    seeds_sorted = sorted(seed_results.keys())

    fig, ax = plt.subplots(figsize=(9, 5))
    x = np.arange(len(metrics_keys))

    for i, seed in enumerate(seeds_sorted):
        vals = [seed_results[seed].get(k, 0) for k in metrics_keys]
        ax.plot(metrics_keys, vals, marker='o', label=f'Seed {seed}', alpha=0.8)

    # Mean + std band
    means = [np.mean([seed_results[s].get(k, 0) for s in seeds_sorted]) for k in metrics_keys]
    stds  = [np.std( [seed_results[s].get(k, 0) for s in seeds_sorted]) for k in metrics_keys]
    ax.plot(metrics_keys, means, 'k--', lw=2, label='Mean')
    ax.fill_between(metrics_keys,
                    [m - s for m, s in zip(means, stds)],
                    [m + s for m, s in zip(means, stds)],
                    color='gray', alpha=0.2, label='±1 std')

    ax.set_ylabel('Score')
    ax.set_title('Ablation C: Seed Stability (Dual Encoder + BM25 Hard-Neg)')
    ax.legend()
    ax.grid(alpha=0.3)
    ax.set_ylim(0, 1.0)
    plt.tight_layout()
    plt.savefig('../results/ablation_seeds.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Need at least 2 seeds to plot.')

## Full Ablation Table

Summary of all ablation experiments with NDCG@10 deltas relative to the default configuration.

In [ ]:
# Aggregate all ablation results into a single table
all_ablations = []

# Baseline
if 'Mean' in ablation_a_results:
    all_ablations.append(('Default (mean, τ=0.05, seed=42)', ablation_a_results['Mean'], True))

# Ablation A
if 'CLS' in ablation_a_results:
    all_ablations.append(('Ablation A: CLS pooling', ablation_a_results['CLS'], False))

# Ablation B
for tau in [0.10, 0.20]:
    if tau in ablation_b_results:
        all_ablations.append((f'Ablation B: τ={tau}', ablation_b_results[tau], False))

# Ablation C (multi-seed)
for seed in [123, 456]:
    if seed in seed_results:
        all_ablations.append((f'Ablation C: seed={seed}', seed_results[seed], False))

if all_ablations:
    metrics_keys = ['ndcg@10', 'recall@10', 'mrr', 'recall@1']
    baseline_ndcg = all_ablations[0][1].get('ndcg@10', 0) if all_ablations else 0

    header = f'{"Configuration":<38} ' + ' '.join(f'{k:>10}' for k in metrics_keys) + '   ΔNDCG'
    print(header)
    print('=' * len(header))

    for label, agg, is_baseline in all_ablations:
        row_vals = ' '.join(f'{agg.get(k, 0):>10.4f}' for k in metrics_keys)
        delta = agg.get('ndcg@10', 0) - baseline_ndcg
        delta_str = '   (base)' if is_baseline else f'   {delta:+.4f}'
        print(f'{label:<38} {row_vals}{delta_str}')
else:
    print('No ablation results yet. Run training commands above.')

## Summary

**Expected ablation findings:**

- **Pooling**: Mean > CLS by ~1–3 NDCG@10 points. Sentence-level mean pooling better captures review semantics than a single CLS token trained for classification.
- **Temperature**: τ=0.05 is near-optimal; higher τ softens gradients and degrades retrieval quality. This matches SimCSE and DPR findings.
- **Seeds**: Std ≈ ±0.003–0.008 NDCG@10 — training is stable; results are not cherry-picked.

→ Proceed to `06_error_analysis.ipynb` for failure case investigation.